# Inferensi Qwen di Google Colab

Notebook ini secara default memulai run baru tanpa meminta ZIP parsial
(`RESTORE_ZIP = False`).

1. Klik **Copy to Drive** pada notebook terbaru ini.
2. Pilih **T4 GPU**, lalu **Run all**.
3. Tunggu pemasangan Ollama, unduhan model, serta prediksi/evaluasi selesai.
4. Simpan **hasil_qwen_lengkap.zip**.

Jika ingin melanjutkan ZIP parsial, ubah `RESTORE_ZIP = True` di sel pertama,
lalu unggah ZIP saat diminta. Checkpoint yang sudah ada pada runtime yang
sama tetap dideteksi dan dilanjutkan; hasil tersimpan tidak dihapus.

Qwen3.5:4b memakai prompt v1 dan agregasi yang sama dengan Gemini, hanya untuk
150 ID uji manual. Ini satu run inferensi tanpa pelatihan: batch dua ulasan,
temperature 0, seed 42, thinking diminta nonaktif, konteks 8.192 token.
Label sentimen uji digunakan hanya pada evaluasi akhir. Ukuran batch yang
berbeda dari Gemini dicatat. Perbaikan ID membatasi jumlah objek dan ID pada
schema; prompt/model/opsi serta jawaban sah sebelumnya tetap dipertahankan.

Checkpoint disimpan setiap batch, ZIP kemajuan ditawarkan setiap 30 ulasan.
Simpan ZIP sebelum mengakhiri runtime; `/content` bersifat sementara. Versi
Ollama, digest model, hash input/kode/prompt dan audit patch diperiksa/dicatat.
Model memakai GPU; bila VRAM tidak digunakan, alur berhenti. Pemasangan hanya
di VM Colab, tanpa Ollama Windows atau kunci API.

Hasil Qwen aktual selesai pada 17 November 2026: F1 pasangan 0,741, macro-F1 sentimen 0,664; 150 ulasan dan 75 batch telah diverifikasi. Uji fixture terpisah dari hasil ini. [Audit](https://github.com/bambangh31/web-mining-absa-kelompok-2/tree/main/results/qwen).
Dokumentasi
tambahan: [Panduan pemulihan](https://github.com/bambangh31/web-mining-absa-kelompok-2/blob/main/docs/project/qwen-pemulihan-id.md).

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import torch

assert torch.cuda.is_available(), "GPU belum tersedia. Pilih runtime T4 GPU dan hubungkan sebelum menjalankan."
import json
import zipfile
from io import BytesIO
from google.colab import files

RESTORE_ZIP = False  # True hanya jika ingin mengunggah ZIP parsial
OUT = Path("/content/hasil_qwen")
MODERN_SOURCE = "30fdae6931f251d174fd4af314722cb3973eced5"
LEGACY_SOURCE = "574fddc5084ff07b6bafe9152b0a029440316157"
RESTORE_CONTENT = None
SOURCE_COMMIT = MODERN_SOURCE
if (OUT / "checkpoint.json").exists() and (OUT / "manifest.json").exists():
    resume_manifest = json.loads((OUT / "manifest.json").read_text())
    SOURCE_COMMIT = resume_manifest["source_commit"]
    assert json.loads((OUT / "checkpoint.json").read_text())["protocol"]["identity"]["source_commit"] == SOURCE_COMMIT
    print("Checkpoint runtime ditemukan; melanjutkan berkas yang sudah ada.")
elif RESTORE_ZIP:
    print("Pilih hasil_qwen_parsial.zip atau ZIP kemajuan yang sudah diunduh.")
    uploads = files.upload()
    assert len(uploads) == 1, "Unggah tepat satu ZIP hasil Qwen."
    RESTORE_CONTENT = next(iter(uploads.values()))
    with zipfile.ZipFile(BytesIO(RESTORE_CONTENT)) as z:
        assert sum(i.file_size for i in z.infolist()) < 30_000_000, "ZIP terlalu besar."
        resume_manifest = json.loads(z.read("manifest.json"))
        checkpoint_info = json.loads(z.read("checkpoint.json"))
        SOURCE_COMMIT = resume_manifest["source_commit"]
        assert checkpoint_info["protocol"]["identity"]["source_commit"] == SOURCE_COMMIT
        print("ZIP memuat", len(checkpoint_info["result"]), "ulasan tersimpan.")
assert SOURCE_COMMIT in [MODERN_SOURCE, LEGACY_SOURCE], "Versi ZIP belum didukung; berkas tidak diubah."
if SOURCE_COMMIT == LEGACY_SOURCE:
    print("Versi lama terdeteksi; perbaikan ID akan dipasang otomatis sebelum prediksi.")
REPO = Path("/content/web-mining-absa-qwen")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/bambangh31/web-mining-absa-kelompok-2.git", str(REPO)], check=True)
assert not subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip(), "Sumber berubah; gunakan folder REPO baru."
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_COMMIT], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, str(REPO))
# Muat ulang modul proyek dari commit yang dipilih, tanpa menghapus berkas hasil.
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]
print("GPU:", torch.cuda.get_device_name(0), "| sumber:", SOURCE_COMMIT)

## 1. Pulihkan berkas dan periksa kesamaan data

In [ ]:
import csv
import json
import zipfile
import shutil
from datetime import datetime, timezone
from google.colab import files
from src.fasttext_vectors import sha256_file
from src.qwen_inference import api_json, json_hash, save_json, infer
from src import llm_label

OUT.mkdir(parents=True, exist_ok=True)
if RESTORE_CONTENT is not None:
    with zipfile.ZipFile(BytesIO(RESTORE_CONTENT)) as z:
        for entry in z.infolist():
            target = (OUT / entry.filename).resolve()
            assert target.is_relative_to(OUT.resolve()), "Path ZIP tidak valid."
            if entry.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            data = z.read(entry)
            assert not target.exists() or target.read_bytes() == data, "Berkas lokal berbeda: gunakan OUT baru."
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)

expected = {
    "data/processed/reviews_clean.csv": "8186e667f08515ca886f42da2fd5d848659f501b5551a6946401c6c68b43af00",
    "data/annotations/split.csv": "67036c5db3ceb2188e4c06c76df4eb1b48199aaa3802c1f323dc14e1b04254ea",
    "data/annotations/verified.csv": "bb3421985df973a57a05f636bc26e60dac68097bf0898a9dbe778ce983fa1df2",
    "data/annotations/final_v1/gold.csv": "8dcea7e41a58d3ac79baa019a5b17c25b173e3094bd0bc9d29a6d3e6a017bade",
}
assert {p: sha256_file(p) for p in expected} == expected, "Input berbeda dari eksperimen sebelumnya."
clean = llm_label.read_clean()
split = llm_label.read_split()
raw = llm_label.label_excel.read_raw("data/raw/com.telkom.tracencare_reviews.csv")
# Hanya ID diambil dari file gold; label sentimen belum digunakan.
with open("data/annotations/final_v1/gold.csv", encoding="utf-8-sig", newline="") as f:
    test_ids = [r["reviewId"] for r in csv.DictReader(f)]
assert len(test_ids) == len(set(test_ids)) == 150
assert all(split[r] == "gold_test" and r in clean and r in raw for r in test_ids)
assert not set(test_ids) & {r for r, s in split.items() if s in ["gold_train", "gold_val", "kalibrasi"]}
items = [(r, clean[r]["text_llm"]) for r in sorted(test_ids)]
originals = {r: raw[r]["content"] for r in test_ids}
expected["data/raw/com.telkom.tracencare_reviews.csv"] = sha256_file("data/raw/com.telkom.tracencare_reviews.csv")
manifest = {"model_code": "B3-Qwen", "source_commit": SOURCE_COMMIT, "n_test": 150,
            "n_train_used": 0, "n_validation_used": 0, "sha256_sources": expected,
            "sha256_code": {p.as_posix(): sha256_file(p) for p in sorted(Path("src").glob("*.py"))},
            "gpu": torch.cuda.get_device_name(0), "python": sys.version,
            "started_utc": datetime.now(timezone.utc).isoformat()}
manifest_path = OUT / "manifest.json"
if manifest_path.exists():
    old = json.loads(manifest_path.read_text())
    for k in ["model_code", "source_commit", "n_test", "sha256_sources", "sha256_code"]:
        assert old[k] == manifest[k], f"Konfigurasi {k} berubah; gunakan OUT baru."
else:
    save_json(manifest_path, manifest)
with (OUT / "id_uji.csv").open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f); writer.writerow(["reviewId"]); writer.writerows([[r] for r in sorted(test_ids)])
print("Input diperiksa: hanya 150 ulasan uji yang akan diprediksi.")

## 2. Pasang Ollama di VM Colab

Versi Ollama dikunci ke 0.40.16. Arsip resmi dan SHA-256 aset rilis diperiksa;
binary serta pustakanya diekstrak ke folder terpisah di `/content`.

In [ ]:
import urllib.request

OLLAMA_VERSION = "0.40.16"
OLLAMA_URL = f"https://github.com/ollama/ollama/releases/download/v{OLLAMA_VERSION}/ollama-linux-amd64.tar.zst"
OLLAMA_SHA256 = "87bfafb2acbc89b2e6a1be0a9043de6aa75fb450696f86bc426e65fc448f788d"
PREFIX = Path("/content/ollama-qwen-runtime")
BINARY = PREFIX / "bin/ollama"
INSTALL_META = PREFIX / "install.json"
ARCHIVE = Path("/content/ollama-linux-amd64.tar.zst")
progress_bucket = [-1]
def progress(blocks, block_size, total):
    done = blocks * block_size
    bucket = done // (128 * 1024 * 1024)
    if bucket != progress_bucket[0]:
        progress_bucket[0] = bucket
        print(f"Unduhan Ollama: {done / 1024**2:.0f} / {total / 1024**2:.0f} MiB", flush=True)

if not (BINARY.exists() and INSTALL_META.exists()):
    if not ARCHIVE.exists():
        temporary = ARCHIVE.with_suffix(".download.part")
        urllib.request.urlretrieve(OLLAMA_URL, temporary, reporthook=progress)
        temporary.replace(ARCHIVE)
    assert sha256_file(ARCHIVE) == OLLAMA_SHA256, "Hash arsip Ollama berbeda; hapus arsip unduhan lalu ulangi."
    if not shutil.which("zstd"):
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"], check=True)
    PREFIX.mkdir(parents=True, exist_ok=True)
    subprocess.run(["tar", "--zstd", "-xf", str(ARCHIVE), "-C", str(PREFIX)], check=True)
    assert BINARY.exists(), "Binary Ollama tidak ditemukan setelah ekstraksi."
    save_json(INSTALL_META, {"version": OLLAMA_VERSION, "archive_sha256": OLLAMA_SHA256, "url": OLLAMA_URL})
install = json.loads(INSTALL_META.read_text())
assert install["version"] == OLLAMA_VERSION and install["archive_sha256"] == OLLAMA_SHA256
save_json(OUT / "ollama_install.json", install)
print("Ollama siap:", BINARY)

## 3. Jalankan server, unduh Qwen, dan periksa penggunaan GPU

In [ ]:
import time

HOST = "http://127.0.0.1:11435"
MODEL = "qwen3.5:4b"
environment = dict(os.environ, OLLAMA_HOST="127.0.0.1:11435",
                   OLLAMA_MODELS="/content/ollama-models-qwen", OLLAMA_NUM_PARALLEL="1",
                   OLLAMA_MAX_LOADED_MODELS="1", OLLAMA_KEEP_ALIVE="30m")
try:
    version = api_json(HOST, "/api/version")
except Exception:
    server_log = (OUT / "ollama_server.log").open("ab")
    server_process = subprocess.Popen([str(BINARY), "serve"], env=environment,
                                      stdout=server_log, stderr=subprocess.STDOUT)
    for _ in range(90):
        try:
            version = api_json(HOST, "/api/version", timeout=2)
            break
        except Exception:
            if server_process.poll() is not None:
                raise RuntimeError("Server Ollama berhenti. Periksa ollama_server.log.")
            time.sleep(1)
    else:
        raise RuntimeError("Server Ollama belum terhubung. Periksa ollama_server.log.")
assert version["version"] == OLLAMA_VERSION, "Versi server berbeda dari versi eksperimen."
subprocess.run([str(BINARY), "pull", MODEL], env=environment, check=True)
tags = api_json(HOST, "/api/tags")["models"]
model_info = next(m for m in tags if m.get("name") == MODEL or m.get("model") == MODEL)
show = api_json(HOST, "/api/show", {"model": MODEL})
save_json(OUT / "ollama_model.json", {"tag": model_info, "show": show, "version": version})
# Pemanasan memakai teks buatan, bukan label atau ulasan uji.
warmup = api_json(HOST, "/api/chat", {"model": MODEL, "stream": False, "think": False,
    "keep_alive": "30m", "options": {"num_ctx": 8192, "num_predict": 8, "temperature": 0},
    "messages": [{"role": "user", "content": "Balas OK."}]}, timeout=300)
assert warmup.get("done"), "Pemanasan model belum selesai."
gpu_state = api_json(HOST, "/api/ps")
running = next(m for m in gpu_state["models"] if m.get("digest") == model_info["digest"])
assert running.get("size_vram", 0) > 0, "Qwen tidak memakai VRAM GPU; periksa runtime/log server."
save_json(OUT / "gpu_sebelum.json", gpu_state)
print(subprocess.check_output([str(BINARY), "ps"], env=environment, text=True))
identity = {"source_commit": SOURCE_COMMIT, "sha256_sources": manifest["sha256_sources"],
            "sha256_code": manifest["sha256_code"], "ollama_version": version["version"],
            "model_digest": model_info["digest"]}
print("Model:", MODEL, "| digest:", model_info["digest"], "| VRAM:", running["size_vram"])

## 4. Prediksi 150 ulasan dan simpan progres

Ini satu run inferensi langsung, bukan tiga seed pelatihan. Setiap batch selesai
menyimpan checkpoint dan CSV prediksi sementara. ZIP kemajuan ditawarkan setiap
30 ulasan. Respons tidak lengkap dicoba ulang; data yang gagal tidak diisi label kosong.

In [ ]:
def export_progress(name="hasil_qwen_kemajuan"):
    path = shutil.make_archive(f"/content/{name}", "zip", OUT)
    print("ZIP:", path, flush=True)
    files.download(path)
    return path

last_export = [0]
def backup_progress(done, total):
    if done - last_export[0] >= 30 or done == total:
        last_export[0] = done
        export_progress(f"hasil_qwen_kemajuan_{done}")
try:
    if SOURCE_COMMIT == LEGACY_SOURCE:
        import hashlib, urllib.request, runpy
        patch_url = "https://raw.githubusercontent.com/bambangh31/web-mining-absa-kelompok-2/30fdae6931f251d174fd4af314722cb3973eced5/src/qwen_resume_patch.py"
        patch_data = urllib.request.urlopen(patch_url).read()
        assert hashlib.sha256(patch_data).hexdigest() == "55a2e617cff78a4349febfe64f03c8a33391520fbcac4a849a932f176cc45df5", "Hash patch berbeda."
        patch_path = OUT / "qwen_resume_patch.py"
        patch_path.write_bytes(patch_data)
        patch_module = runpy.run_path(str(patch_path))
        patch_module["activate"](OUT)
    predictions = infer(items, originals, OUT, identity, host=HOST, model=MODEL,
                        batch_size=2, on_progress=backup_progress)
    assert len(predictions) == 150 and set(predictions) == set(test_ids)
except Exception as exc:
    save_json(OUT / "error.json", {"error": str(exc), "time": datetime.now(timezone.utc).isoformat()})
    export_progress("hasil_qwen_parsial")
    raise
save_json(OUT / "gpu_sesudah.json", api_json(HOST, "/api/ps"))
print("Prediksi lengkap. Sel berikut mengevaluasi dengan label manual.")

## 5. Evaluasi akhir dan unduh ZIP lengkap

Label sentimen uji baru dipakai di sini. Evaluasi membandingkan Qwen dengan
13 run terdahulu (tujuh metode). Tidak melatih ulang BiLSTM/IndoBERT.
Jangan mengubah prompt atau opsi berdasarkan skor uji.

In [ ]:
from src.evaluate import evaluate, load_labels

path = OUT / "predictions/B3-Qwen_42.csv"
pred = load_labels(path)
with path.open(encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))
assert len(rows) == len(pred) == 150 and set(pred) == set(test_ids)
gold = load_labels("data/annotations/final_v1/gold.csv")
assert set(gold) == set(test_ids)
score = evaluate(gold, pred, test_ids)
save_json(OUT / "metrik_qwen.json", score)
specs = ["B1-LR=results/predictions/B1-LR_42.csv", "B1-SVM=results/predictions/B1-SVM_42.csv",
         "B1-NB=results/predictions/B1-NB_42.csv", "B3-Gemini=data/annotations/llm_gemini-3.5-flash-lite.csv"]
specs += [f"{code}_{s}=results/predictions/{code}_{s}.csv" for code in ["B2", "B2-L", "B4"] for s in [42,43,44]]
specs += [f"B3-Qwen={path}"]
command = [sys.executable, "src/evaluate.py", "--gold", "data/annotations/final_v1/gold.csv",
           "--ids-split", "gold_test", "--out", str(OUT / "evaluasi")]
for spec in specs:
    command.extend(["--pred", spec])
subprocess.run(command, check=True)
save_json(OUT / "completion.json", {"status": "completed", "n_test": 150, "model": MODEL,
    "model_digest": identity["model_digest"], "finished_utc": datetime.now(timezone.utc).isoformat()})
print("F1 pasangan Qwen:", score["f1_pasangan"])
export_progress("hasil_qwen_lengkap")
print("Simpan hasil_qwen_lengkap.zip dan berikan untuk verifikasi serta pembaruan laporan.")

## Unduh ulang hasil saat ini bila diperlukan

Sel ini hanya mengunduh isi folder hasil saat ini. Sebelum evaluasi selesai,
ZIP masih parsial dan bukan hasil lengkap. Log server, respons, serta checkpoint
disertakan; bobot Qwen dan binary Ollama tidak dimasukkan ke ZIP.

In [ ]:
export_progress("hasil_qwen_saat_ini")